In [ ]:
import urllib.request

files = {
    "EDITIONS": "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz",
    "WORKS": "https://openlibrary.org/data/ol_dump_works_latest.txt.gz",
    "AUTHORS": "https://openlibrary.org/data/ol_dump_authors_latest.txt.gz",
}

for name, url in files.items():
    print(f"\n--- {name} ---")
    req = urllib.request.Request(url, method="HEAD")
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            size = r.headers.get("Content-Length")
            print("HTTP :", r.status)
            print("Taille :", round(int(size)/1024**3, 2), "GB" if size else "inconnue")
            print("Type :", r.headers.get("Content-Type"))
    except Exception as e:
        print("Erreur :", str(e))

In [ ]:
from huggingface_hub import hf_hub_upload, hf_hub_download
from kaggle_secrets import UserSecretsClient
import os, hashlib

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO_ID = "Moses-team/nexus-science-backup"

V10_BACKUP_DIR = "v10_recovery_v10_3"

commit, stage_path = v10_read_commit()
stage_name = commit["stage_filename"]
remote_stage = f"{V10_BACKUP_DIR}/{stage_name}"

hf_hub_upload(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    path_or_fileobj=stage_path,
    path_in_repo=remote_stage,
    token=HF_TOKEN
)

print("=== TRANSFERT STAGE V10.3 ===")
print("Génération :", commit["generation"])
print("Fichier :", remote_stage)
print("Taille locale :", os.path.getsize(stage_path))
print("STATUT : STAGE ENVOYE")


# Bibliothèque Scientifique — Nexus Science
## Phase 1 : Collecte et filtrage Open Library
Environnement cloud Kaggle — traitement en flux

In [ ]:
import os, gzip, json, urllib.request

BASE = "/kaggle/working/nexus_science"
os.makedirs(BASE, exist_ok=True)

print("Dossier de travail :", BASE)
print("Environnement prêt pour le traitement Open Library.")

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
subjects = collections.Counter()
languages = collections.Counter()
years = collections.Counter()
count = 0
valid = 0

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            count += 1

            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                valid += 1

                for s in rec.get("subjects", []):
                    if isinstance(s, str):
                        subjects[s.lower()] += 1

                for lang in rec.get("languages", []):
                    if isinstance(lang, dict):
                        languages[lang.get("key", "")] += 1

                date = str(rec.get("publish_date", ""))
                if len(date) >= 4 and date[:4].isdigit():
                    years[date[:4]] += 1

            except Exception:
                pass

print("Lignes examinées :", count)
print("Notices JSON valides :", valid)
print("\nTOP 40 SUJETS :")
for s, n in subjects.most_common(40):
    print(n, "|", s)

print("\nTOP LANGUES :")
for s, n in languages.most_common(20):
    print(n, "|", s)

print("\nDATES les plus fréquentes :")
for s, n in years.most_common(20):
    print(n, "|", s)

In [ ]:
SCIENCE_KEYWORDS = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "biological",
    "medicine", "medical", "health", "anatomy",
    "engineering", "technology", "computer science",
    "astronomy", "astrophysics", "geology", "geography",
    "ecology", "environment", "botany", "zoology",
    "statistics", "research", "laboratory", "microbiology",
    "genetics", "pharmacology", "neuroscience", "agriculture",
    "economics", "psychology", "sociology", "education",
    "philosophy of science", "nursing", "veterinary"
]

print("Filtre scientifique initialisé.")
print("Nombre de mots-clés :", len(SCIENCE_KEYWORDS))
print("Période cible : 1720–2026")
print("Langues cibles : multilingues")

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
count = 0
selected = 0
matched_subjects = collections.Counter()
examples = []

keywords = [k.lower() for k in SCIENCE_KEYWORDS]

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                count += 1

                title = str(rec.get("title", "")).lower()
                subjects = rec.get("subjects", [])
                subject_text = " ".join(
                    s.lower() for s in subjects if isinstance(s, str)
                )
                text_to_check = title + " " + subject_text

                if any(k in text_to_check for k in keywords):
                    selected += 1
                    for s in subjects:
                        if isinstance(s, str):
                            matched_subjects[s] += 1

                    if len(examples) < 15:
                        examples.append(rec.get("title", "Sans titre"))

            except Exception:
                pass

print("Notices examinées :", count)
print("Notices sélectionnées :", selected)
print("Proportion :", round(selected/count*100, 2) if count else 0, "%")
print("\nEXEMPLES :")
for title in examples:
    print("-", title)

print("\nSUJETS RETENUS :")
for subject, n in matched_subjects.most_common(20):
    print(n, "|", subject)

In [ ]:
import urllib.request, zlib, json, collections

url = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
req = urllib.request.Request(url, headers={"Range": "bytes=0-67108863"})

d = zlib.decompressobj(16 + zlib.MAX_WBITS)
buffer = b""
count = selected = excluded = 0
examples = []

science = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "medicine", "medical",
    "anatomy", "engineering", "computer science", "astronomy",
    "geology", "ecology", "botany", "zoology", "statistics",
    "microbiology", "genetics", "pharmacology", "neuroscience",
    "agriculture", "nursing", "veterinary", "technology",
    "environmental science", "biotechnology", "physics",
    "natural science", "research methodology"
]

noise = [
    "statutory instrument", "parliamentary debates",
    "parliamentary proceedings", "bill [", "regulations",
    "establishment order", "amendment order",
    "house of lords", "government notice"
]

with urllib.request.urlopen(req, timeout=120) as r:
    while count < 3000:
        chunk = r.read(256 * 1024)
        if not chunk:
            break

        buffer += d.decompress(chunk)

        while b"\n" in buffer and count < 3000:
            line, buffer = buffer.split(b"\n", 1)
            parts = line.split(b"\t")
            if len(parts) < 4:
                continue

            try:
                rec = json.loads(parts[-1])
                count += 1

                title = str(rec.get("title", "")).lower()
                subjects = [
                    s.lower() for s in rec.get("subjects", [])
                    if isinstance(s, str)
                ]
                searchable = title + " " + " ".join(subjects)

                relevant = any(k in searchable for k in science)
                administrative = any(k in title for k in noise)

                if relevant and not administrative:
                    selected += 1
                    if len(examples) < 25:
                        examples.append(title)
                elif relevant and administrative:
                    excluded += 1

            except Exception:
                pass

print("Notices examinées :", count)
print("Retenues V2 :", selected)
print("Documents administratifs écartés :", excluded)
print("\nEXEMPLES RETENUS :")
for t in examples:
    print("-", t)

In [ ]:
# NEXUS SCIENCE — GLOBAL OPEN LIBRARY EDITIONS SCANNER V1
# Streaming + checkpoint + restart-safe
# Aucune écriture dans Neon.

import os, json, gzip, urllib.request, time, re

BASE = "/kaggle/working/nexus_science"
os.makedirs(BASE, exist_ok=True)

URL = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
OUT = os.path.join(BASE, "scientific_editions.jsonl")
CHECKPOINT = os.path.join(BASE, "scanner_checkpoint.json")

KEYWORDS = [
    "science", "scientific", "mathematics", "mathematical",
    "physics", "chemistry", "biology", "biological",
    "medicine", "medical", "health", "anatomy",
    "engineering", "technology", "computer science",
    "astronomy", "astrophysics", "geology", "geography",
    "ecology", "environment", "botany", "zoology",
    "statistics", "research", "laboratory", "microbiology",
    "genetics", "pharmacology", "neuroscience", "agriculture",
    "economics", "psychology", "sociology", "education",
    "nursing", "veterinary", "biotechnology", "mathematics",
    "matematica", "ciencia", "ciencias", "biologia",
    "fisica", "quimica", "medicina", "ingenieria",
    "astronomia", "mathematiques", "scientifique",
    "biologie", "physique", "chimie", "médecine",
    "matemática", "ciência", "biologia", "medicina",
    "engenharia", "astronomia"
]

# Termes fortement administratifs : exclusions prudentes sur le titre seulement
NOISE = [
    "statutory instrument", "parliamentary debates",
    "parliamentary proceedings", "government notice",
    "house of lords bill"
]

def matches(record):
    title = str(record.get("title", "")).lower()
    subjects = record.get("subjects", [])
    if not isinstance(subjects, list):
        subjects = []

    subject_text = " ".join(
        s.lower() for s in subjects if isinstance(s, str)
    )
    searchable = title + " " + subject_text

    relevant = any(k in searchable for k in KEYWORDS)
    excluded = any(k in title for k in NOISE)

    return relevant and not excluded

# Reprise du compteur de lignes
resume_line = 0
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, "r", encoding="utf-8") as f:
        state = json.load(f)
        resume_line = state.get("scan_lines", 0)

print("Scanner Nexus Science")
print("Lignes déjà parcourues :", resume_line)
print("Fichier de sortie :", OUT)
print("Mode : flux gzip / checkpoint")

scan_lines = resume_line
valid = 0
selected = 0
started = time.time()
last_checkpoint = scan_lines

# append pour préserver les résultats précédents
with open(OUT, "a", encoding="utf-8") as output:
    request = urllib.request.Request(
        URL, headers={"User-Agent": "NexusScience-Research/1.0"}
    )

    with urllib.request.urlopen(request, timeout=120) as response:
        with gzip.GzipFile(fileobj=response, mode="rb") as gz:

            current_line = 0

            for raw_line in gz:
                current_line += 1

                # Relecture du début pour reprendre le checkpoint
                if current_line <= resume_line:
                    if current_line % 1000000 == 0:
                        print("Reprise :", current_line)
                    continue

                scan_lines += 1

                parts = raw_line.rstrip(b"\n").split(b"\t")
                if len(parts) < 4:
                    continue

                try:
                    record = json.loads(parts[-1])
                    valid += 1

                    if matches(record):
                        output.write(json.dumps(
                            record, ensure_ascii=False
                        ) + "\n")
                        selected += 1

                except Exception:
                    pass

                if scan_lines - last_checkpoint >= 10000:
                    output.flush()

                    with open(CHECKPOINT, "w", encoding="utf-8") as cp:
                        json.dump(
                            {"scan_lines": scan_lines},
                            cp
                        )

                    last_checkpoint = scan_lines
                    elapsed = round(time.time() - started, 1)

                    print(
                        "Progression lignes :", scan_lines,
                        "| JSON valides :", valid,
                        "| retenues :", selected,
                        "| secondes :", elapsed
                    )

# Dernier checkpoint
with open(CHECKPOINT, "w", encoding="utf-8") as cp:
    json.dump({"scan_lines": scan_lines}, cp)

print("\nSCAN TERMINÉ")
print("Lignes parcourues :", scan_lines)
print("JSON valides :", valid)
print("Retenues pendant cette session :", selected)
print("Temps (sec) :", round(time.time() - started, 1))
print("Résultats :", OUT)

In [ ]:
!ls -lh /kaggle/working/nexus_science

In [ ]:
import json

with open("/kaggle/working/nexus_science/scanner_checkpoint.json", "r") as f:
    print(json.load(f))

In [ ]:
import os

p = "/kaggle/working/nexus_science/scientific_editions.jsonl"

print("Fichier présent :", os.path.exists(p))
print("Taille (Go) :", round(os.path.getsize(p) / 1024**3, 3))

with open(p, "rb") as f:
    f.seek(max(0, os.path.getsize(p) - 2000))
    print("\nFIN DU FICHIER :\n")
    print(f.read().decode("utf-8", errors="replace"))

In [25]:
import os

p = "/kaggle/working/nexus_science/scientific_editions.jsonl"

print("Fichier présent :", os.path.exists(p))
if os.path.exists(p):
    print("Taille (Go) :", round(os.path.getsize(p) / 1024**3, 3))

Fichier présent : True
Taille (Go) : 6.523


In [1]:
from pathlib import Path

base = Path("/kaggle/working/nexus_science")

print("=== SAUVEGARDE NEXUS : INVENTAIRE ===")

if base.exists():
    for p in sorted(base.rglob("*")):
        if p.is_file():
            print(f"{p.name} | {p.stat().st_size:,} octets | {p}")
else:
    print("Le dossier nexus_science est absent.")

print("\n=== AUTRES FICHIERS VOLUMINEUX ===")

root = Path("/kaggle/working")
for p in root.iterdir():
    if p.is_file():
        print(f"{p.name} | {round(p.stat().st_size / 1024**3, 3)} Go")

=== SAUVEGARDE NEXUS : INVENTAIRE ===
Le dossier nexus_science est absent.

=== AUTRES FICHIERS VOLUMINEUX ===


In [2]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/temp"),
    Path("/tmp"),
]

print("=== RECHERCHE DE FICHIERS NEXUS ===")

for root in roots:
    if not root.exists():
        continue

    print(f"\nDossier : {root}")

    for p in root.rglob("*"):
        try:
            if p.is_file() and p.stat().st_size > 100_000_000:
                size = p.stat().st_size / 1024**3
                print(f"{size:.3f} Go | {p}")
        except (PermissionError, OSError):
            pass

=== RECHERCHE DE FICHIERS NEXUS ===

Dossier : /kaggle/working

Dossier : /tmp


In [3]:
import os, shutil

print("=== ENVIRONNEMENT NEXUS ===")
print("Répertoire :", os.getcwd())
print("Espace libre :", round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2), "Go")
print("Espace total :", round(shutil.disk_usage("/kaggle/working").total / 1024**3, 2), "Go")
print("Python :", os.sys.version)

=== ENVIRONNEMENT NEXUS ===
Répertoire : /kaggle/working
Espace libre : 19.5 Go
Espace total : 19.52 Go
Python : 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


In [4]:
import os

base = "/kaggle/working/nexus_science"

print("=== ÉTAT DES SAUVEGARDES NEXUS ===")

if os.path.exists(base):
    for f in os.listdir(base):
        p = os.path.join(base, f)
        if os.path.isfile(p):
            print(f"{f} | {round(os.path.getsize(p)/1024**2, 2)} Mo")
        else:
            print(f"{f} | dossier")
else:
    print("Dossier Nexus absent.")

print("\n=== CONFIGURATION ===")
print("RUN_FULL_SCAN =", RUN_FULL_SCAN)

=== ÉTAT DES SAUVEGARDES NEXUS ===
Dossier Nexus absent.

=== CONFIGURATION ===


NameError: name 'RUN_FULL_SCAN' is not defined

In [5]:
import os

base = "/kaggle/working/nexus_science"

print("=== ÉTAT DES SAUVEGARDES NEXUS ===")

if os.path.exists(base):
    for f in os.listdir(base):
        p = os.path.join(base, f)
        if os.path.isfile(p):
            print(f"{f} | {round(os.path.getsize(p)/1024**2, 2)} Mo")
        else:
            print(f"{f} | dossier")
else:
    print("Dossier Nexus absent.")

print("\n=== ÉTAT DU SCANNER ===")
print("Cette cellule n'exécute aucun scanner.")

=== ÉTAT DES SAUVEGARDES NEXUS ===
Dossier Nexus absent.

=== ÉTAT DU SCANNER ===
Cette cellule n'exécute aucun scanner.


In [6]:
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"

CHUNKS.mkdir(parents=True, exist_ok=True)

print("Dossier V3 :", BASE)
print("Dossier segments :", CHUNKS)
print("Dossiers créés :", BASE.exists() and CHUNKS.exists())

Dossier V3 : /kaggle/working/nexus_science_v3
Dossier segments : /kaggle/working/nexus_science_v3/chunks
Dossiers créés : True


In [7]:
import json, os
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

BASE.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

manifest = {
    "version": 3,
    "project": "Bibliothèque Scientifique — Nexus Science",
    "source": "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz",
    "status": "prepared",
    "scan_lines": 0,
    "valid_json": 0,
    "selected": 0,
    "invalid_json": 0,
    "out_of_period": 0,
    "completed_chunks": [],
    "current_chunk": None,
    "updated_at_utc": None
}

tmp = MANIFEST.with_suffix(".tmp")

with open(tmp, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
    f.flush()
    os.fsync(f.fileno())

os.replace(tmp, MANIFEST)

# Vérification
loaded = json.loads(MANIFEST.read_text(encoding="utf-8"))

print("=== MANIFESTE V3 ===")
print("Version :", loaded["version"])
print("Statut :", loaded["status"])
print("Segments validés :", len(loaded["completed_chunks"]))
print("Manifeste vérifié :", loaded["version"] == 3)
print("Fichier :", MANIFEST)

=== MANIFESTE V3 ===
Version : 3
Statut : prepared
Segments validés : 0
Manifeste vérifié : True
Fichier : /kaggle/working/nexus_science_v3/manifest_v3.json


In [11]:
import hashlib
import json
import tempfile
from pathlib import Path

def validate_chunk(path, expected_count, expected_sha256):
    path = Path(path)
    count = 0

    with open(path, "rb") as f:
        for line in f:
            json.loads(line)
            count += 1

    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    return {
        "valid_json": count == expected_count,
        "valid_hash": digest == expected_sha256,
        "records": count,
        "sha256": digest
    }

# Test isolé : aucun fichier Nexus réel n'est modifié.
with tempfile.TemporaryDirectory(prefix="nexus_v3_test_") as tmp:
    test_dir = Path(tmp)
    test_file = test_dir / "chunk_000001.jsonl"

    records = [
        {"title": "Physics", "year": 1900},
        {"title": "Mathematics", "year": 1920},
        {"title": "Biology", "year": 2020}
    ]

    with open(test_file, "wb") as f:
        for record in records:
            f.write(
                json.dumps(record, ensure_ascii=False).encode("utf-8")
                + b"\n"
            )

    digest = hashlib.sha256(test_file.read_bytes()).hexdigest()
    result = validate_chunk(test_file, len(records), digest)

    print("=== TEST MOTEUR V3 ===")
    print("Notices :", result["records"])
    print("JSON valides :", result["valid_json"])
    print("SHA-256 vérifié :", result["valid_hash"])
    print("Test réussi :", result["valid_json"] and result["valid_hash"])

=== TEST MOTEUR V3 ===
Notices : 3
JSON valides : True
SHA-256 vérifié : True
Test réussi : True


In [12]:
import hashlib
import json
import os
from pathlib import Path

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def atomic_json_write(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def finalize_chunk(chunk_path, records_count):
    chunk_path = Path(chunk_path)

    # Contrôle des données avant validation
    with open(chunk_path, "rb") as f:
        count = 0
        for line in f:
            json.loads(line)
            count += 1

    if count != records_count:
        raise ValueError("Le nombre de notices ne correspond pas.")

    return {
        "file": chunk_path.name,
        "records": count,
        "bytes": chunk_path.stat().st_size,
        "sha256": sha256_file(chunk_path)
    }

# Simulation isolée
with tempfile.TemporaryDirectory(prefix="nexus_v3_chunks_") as tmp:
    folder = Path(tmp)
    results = []

    for index, records in enumerate([
        [{"title": "Physics"}, {"title": "Biology"}],
        [{"title": "Chemistry"}, {"title": "Astronomy"}]
    ], start=1):

        chunk = folder / f"chunk_{index:06d}.jsonl"

        with open(chunk, "wb") as f:
            for record in records:
                f.write(
                    json.dumps(record, ensure_ascii=False).encode("utf-8")
                    + b"\n"
                )

        metadata = finalize_chunk(chunk, len(records))
        results.append(metadata)

    test_manifest = {
        "version": 3,
        "status": "test",
        "completed_chunks": results
    }

    atomic_json_write(folder / "manifest_test.json", test_manifest)

    verified = json.loads(
        (folder / "manifest_test.json").read_text(encoding="utf-8")
    )

    print("=== GESTIONNAIRE V3 ===")
    print("Segments :", len(verified["completed_chunks"]))
    print("Notices :", sum(c["records"] for c in verified["completed_chunks"]))
    print("Taille cumulée :", sum(c["bytes"] for c in verified["completed_chunks"]))
    print("SHA-256 présents :", all(c["sha256"] for c in verified["completed_chunks"]))
    print("Manifeste relu :", len(verified["completed_chunks"]) == 2)
    print("TEST :", "OK" if len(verified["completed_chunks"]) == 2 and all(c["sha256"] for c in verified["completed_chunks"]) else "ÉCHEC")

=== GESTIONNAIRE V3 ===
Segments : 2
Notices : 4
Taille cumulée : 88
SHA-256 présents : True
Manifeste relu : True
TEST : OK


In [13]:
import os
import json
import hashlib
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

BASE.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 100_000

def atomic_json_write(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()

def validate_chunk(path, expected_count):
    path = Path(path)
    count = 0

    with open(path, "rb") as f:
        for line in f:
            json.loads(line)
            count += 1

    if count != expected_count:
        raise ValueError(
            f"Nombre incorrect : {count} au lieu de {expected_count}"
        )

    return {
        "records": count,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path)
    }

def load_manifest():
    if not MANIFEST.exists():
        raise FileNotFoundError("Manifeste V3 absent.")

    data = json.loads(MANIFEST.read_text(encoding="utf-8"))

    if data.get("version") != 3:
        raise ValueError("Version du manifeste incompatible.")

    return data

def register_chunk(path, metadata):
    manifest = load_manifest()

    name = Path(path).name

    # Empêcher l'enregistrement en double
    if any(c["file"] == name for c in manifest["completed_chunks"]):
        raise ValueError(f"Segment déjà enregistré : {name}")

    entry = {
        "file": name,
        "records": metadata["records"],
        "bytes": metadata["bytes"],
        "sha256": metadata["sha256"]
    }

    manifest["completed_chunks"].append(entry)
    manifest["selected"] += metadata["records"]
    manifest["status"] = "running"

    atomic_json_write(MANIFEST, manifest)

    return entry

class NexusChunkWriter:

    def __init__(self, chunk_size=CHUNK_SIZE):
        self.chunk_size = chunk_size
        self.count = 0
        self.chunk_index = 0
        self.file = None
        self.path = None

    def open_chunk(self):
        self.chunk_index += 1

        self.path = CHUNKS / (
            f"chunk_{self.chunk_index:06d}.jsonl"
        )

        if self.path.exists():
            raise FileExistsError(
                f"Le segment existe déjà : {self.path}"
            )

        self.file = open(self.path, "xb")
        self.count = 0

    def write_record(self, record):
        if self.file is None:
            self.open_chunk()

        payload = (
            json.dumps(record, ensure_ascii=False)
            .encode("utf-8") + b"\n"
        )

        self.file.write(payload)
        self.count += 1

        if self.count >= self.chunk_size:
            self.finalize()

    def finalize(self):
        if self.file is None:
            return None

        self.file.flush()
        os.fsync(self.file.fileno())
        self.file.close()
        self.file = None

        metadata = validate_chunk(self.path, self.count)

        entry = register_chunk(self.path, metadata)

        print(
            "SEGMENT CERTIFIÉ :",
            entry["file"],
            "| notices :",
            entry["records"],
            "| SHA-256 :",
            entry["sha256"][:16]
        )

        self.path = None
        self.count = 0

        return entry

    def close(self):
        return self.finalize()

print("=== MOTEUR V3 INSTALLÉ ===")
print("Segment maximal :", CHUNK_SIZE, "notices")
print("Validation JSON : active")
print("SHA-256 : actif")
print("Manifeste atomique : actif")
print("Téléchargement : désactivé")
print("Neon : aucune connexion")

=== MOTEUR V3 INSTALLÉ ===
Segment maximal : 100000 notices
Validation JSON : active
SHA-256 : actif
Manifeste atomique : actif
Téléchargement : désactivé
Neon : aucune connexion


In [14]:
import json
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

# Vérifier que notre espace de test est propre
manifest_before = json.loads(MANIFEST.read_text(encoding="utf-8"))

if manifest_before["completed_chunks"] or manifest_before["selected"] != 0:
    raise RuntimeError("Le manifeste n'est pas vierge. Test refusé.")

if list(CHUNKS.iterdir()):
    raise RuntimeError("Le dossier chunks n'est pas vide. Test refusé.")

test_records = [
    {"title": "Physics", "year": 1900},
    {"title": "Biology", "year": 1920},
    {"title": "Chemistry", "year": 2020}
]

writer = NexusChunkWriter(chunk_size=3)

try:
    for record in test_records:
        writer.write_record(record)

    # Le 3e enregistrement déclenche déjà la certification.
    result = load_manifest()

    assert len(result["completed_chunks"]) == 1
    assert result["selected"] == 3

    entry = result["completed_chunks"][0]
    chunk_path = CHUNKS / entry["file"]

    verified = validate_chunk(chunk_path, 3)

    assert verified["sha256"] == entry["sha256"]
    assert verified["bytes"] == entry["bytes"]

    print("=== TEST V3 RÉEL ===")
    print("Segment :", entry["file"])
    print("Notices :", verified["records"])
    print("SHA-256 concordant :", True)
    print("Manifeste :", True)
    print("TEST :", "OK")

finally:
    # Nettoyage du test et retour au manifeste initial
    if writer.file is not None:
        writer.file.close()

    for f in CHUNKS.iterdir():
        if f.is_file():
            f.unlink()

    atomic_json_write(MANIFEST, manifest_before)

print("\n=== ÉTAT APRÈS NETTOYAGE ===")
print("Segments :", len(load_manifest()["completed_chunks"]))
print("Notices :", load_manifest()["selected"])
print("Dossier de segments vide :", not list(CHUNKS.iterdir()))

SEGMENT CERTIFIÉ : chunk_000001.jsonl | notices : 3 | SHA-256 : 81f181d182d8b1dc
=== TEST V3 RÉEL ===
Segment : chunk_000001.jsonl
Notices : 3
SHA-256 concordant : True
Manifeste : True
TEST : OK

=== ÉTAT APRÈS NETTOYAGE ===
Segments : 0
Notices : 0
Dossier de segments vide : True


In [15]:
import json, tempfile
from pathlib import Path

def simulate_recovery_protocol():
    with tempfile.TemporaryDirectory(prefix="nexus_v3_recovery_") as tmp:
        root = Path(tmp)
        chunk = root / "chunk_000001.jsonl"

        records = [
            {"title": "Physics"},
            {"title": "Biology"}
        ]

        with open(chunk, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")

        metadata = finalize_chunk(chunk, 2)

        source = {
            "url": "https://example.org/dump.gz",
            "etag": "test-etag-001",
            "length": "1000"
        }

        checkpoint = {
            "version": 3,
            "source": source,
            "scan_lines": 5000,
            "completed_chunks": [{
                "file": chunk.name,
                "records": metadata["records"],
                "bytes": metadata["bytes"],
                "sha256": metadata["sha256"],
                "source_lines": 5000
            }]
        }

        cp = root / "checkpoint.json"
        atomic_json_write(cp, checkpoint)

        recovered = json.loads(cp.read_text(encoding="utf-8"))

        # 1. Contrôle de source
        source_ok = recovered["source"] == source

        # 2. Validation segment
        segment = root / recovered["completed_chunks"][0]["file"]
        segment_ok = (
            validate_chunk(segment, 2)["sha256"]
            == recovered["completed_chunks"][0]["sha256"]
        )

        # 3. Refus d'une autre source
        altered_source = dict(source)
        altered_source["etag"] = "different-etag"

        source_change_refused = altered_source != recovered["source"]

        # 4. Progression récupérée
        recovered_lines = recovered["scan_lines"]
        next_chunk = len(recovered["completed_chunks"]) + 1

        return {
            "source_ok": source_ok,
            "segment_ok": segment_ok,
            "source_change_refused": source_change_refused,
            "recovered_lines": recovered_lines,
            "next_chunk": next_chunk
        }

result = simulate_recovery_protocol()

print("=== TEST PROTOCOLE DE REPRISE ===")
print("Identité source :", result["source_ok"])
print("Segment certifié :", result["segment_ok"])
print("Source différente refusée :", result["source_change_refused"])
print("Lignes récupérées :", result["recovered_lines"])
print("Prochain segment :", result["next_chunk"])
print("TEST :", "OK" if all([
    result["source_ok"],
    result["segment_ok"],
    result["source_change_refused"],
    result["recovered_lines"] == 5000,
    result["next_chunk"] == 2
]) else "ÉCHEC")

=== TEST PROTOCOLE DE REPRISE ===
Identité source : True
Segment certifié : True
Source différente refusée : True
Lignes récupérées : 5000
Prochain segment : 2
TEST : OK


In [16]:
import json, hashlib, os
from pathlib import Path

BASE = Path("/kaggle/working/nexus_science_v3")
CHUNKS = BASE / "chunks"
MANIFEST = BASE / "manifest_v3.json"

def v3_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_atomic_write(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def v3_load_manifest():
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def v3_next_chunk_number():
    manifest = v3_load_manifest()
    numbers = []

    for entry in manifest.get("completed_chunks", []):
        name = entry["file"]
        if name.startswith("chunk_") and name.endswith(".jsonl"):
            numbers.append(int(name[6:12]))

    return max(numbers, default=0) + 1

def v3_verify_all_chunks():
    manifest = v3_load_manifest()
    verified = 0

    for entry in manifest.get("completed_chunks", []):
        path = CHUNKS / entry["file"]

        if not path.exists():
            raise FileNotFoundError(entry["file"])

        if path.stat().st_size != entry["bytes"]:
            raise ValueError("Taille incohérente : " + entry["file"])

        if v3_hash(path) != entry["sha256"]:
            raise ValueError("SHA-256 incohérent : " + entry["file"])

        verified += 1

    return verified

print("=== GESTIONNAIRE INTÉGRÉ V3 ===")
print("Prochain segment :", v3_next_chunk_number())
print("Segments vérifiés :", v3_verify_all_chunks())
print("Manifeste :", MANIFEST)
print("Scan massif : non lancé")

=== GESTIONNAIRE INTÉGRÉ V3 ===
Prochain segment : 1
Segments vérifiés : 0
Manifeste : /kaggle/working/nexus_science_v3/manifest_v3.json
Scan massif : non lancé


In [18]:
import gzip, json, tempfile, hashlib, os
from pathlib import Path

def run_v3_local_stream_test():
    with tempfile.TemporaryDirectory(prefix="nexus_v3_stream_") as tmp:
        root = Path(tmp)
        source = root / "mini_dump.gz"
        outdir = root / "chunks"
        outdir.mkdir()

        # Mini-dump artificiel de 20 notices
        records = [
            {"title": f"Physics Book {i}", "publish_date": str(1900 + i)}
            for i in range(20)
        ]

        with gzip.open(source, "wb") as gz:
            for i, record in enumerate(records):
                line = f"/works/OL{i}M\tedition\t".encode() + json.dumps(record).encode() + b"\n"
                gz.write(line)

        # Lecture du vrai fichier gzip
        parsed = 0
        selected = []

        with gzip.open(source, "rb") as gz:
            for line in gz:
                parts = line.rstrip(b"\r\n").split(b"\t")
                record = json.loads(parts[-1])
                parsed += 1
                selected.append(record)

        # Création et certification d'un segment local
        chunk = outdir / "chunk_000001.jsonl"
        with open(chunk, "wb") as f:
            for record in selected[:10]:
                f.write(json.dumps(record).encode() + b"\n")

        digest = hashlib.sha256(chunk.read_bytes()).hexdigest()

        # Checkpoint simulé après 10 lignes
        checkpoint = {
            "version": 3,
            "scan_lines": 10,
            "selected": 10,
            "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
            "completed_chunks": [{
                "file": chunk.name,
                "records": 10,
                "bytes": chunk.stat().st_size,
                "sha256": digest
            }]
        }

        cp = root / "checkpoint.json"
        with open(cp, "w", encoding="utf-8") as f:
            json.dump(checkpoint, f)

        recovered = json.loads(cp.read_text())

        # Vérification du segment et du checkpoint
        verified_hash = hashlib.sha256(chunk.read_bytes()).hexdigest()
        segment_ok = verified_hash == recovered["completed_chunks"][0]["sha256"]

        # Relecture du dump et saut des 10 lignes déjà checkpointées
        resumed_count = 0
        with gzip.open(source, "rb") as gz:
            for line_number, line in enumerate(gz, start=1):
                if line_number <= recovered["scan_lines"]:
                    continue
                json.loads(line.rstrip(b"\r\n").split(b"\t")[-1])
                resumed_count += 1

        return {
            "source_lines": parsed,
            "checkpoint_lines": recovered["scan_lines"],
            "segment_ok": segment_ok,
            "resumed_lines": resumed_count,
            "test_ok": parsed == 20 and segment_ok and resumed_count == 10
        }

test = run_v3_local_stream_test()

print("=== TEST FLUX GZIP V3 ===")
print("Lignes mini-dump :", test["source_lines"])
print("Lignes checkpointées :", test["checkpoint_lines"])
print("Segment vérifié :", test["segment_ok"])
print("Lignes après reprise simulée :", test["resumed_lines"])
print("TEST :", "OK" if test["test_ok"] else "ÉCHEC")

=== TEST FLUX GZIP V3 ===
Lignes mini-dump : 20
Lignes checkpointées : 10
Segment vérifié : True
Lignes après reprise simulée : 10
TEST : OK


In [19]:
import json, hashlib, tempfile
from pathlib import Path

def test_interruption_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_crash_v3_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        records = [
            {"id": i, "title": f"Science {i}"}
            for i in range(1, 31)
        ]

        chunk_size = 10

        # 1. Premier segment certifié
        first = records[:10]
        chunk1 = chunks / "chunk_000001.jsonl"

        with open(chunk1, "w", encoding="utf-8") as f:
            for r in first:
                f.write(json.dumps(r) + "\n")

        digest1 = hashlib.sha256(chunk1.read_bytes()).hexdigest()

        committed_line = 10

        # 2. Simuler une panne après 17 lignes
        interrupted_records = records[10:17]

        partial = chunks / "chunk_000002.partial"

        with open(partial, "w", encoding="utf-8") as f:
            for r in interrupted_records:
                f.write(json.dumps(r) + "\n")

        # 3. Panne : le segment partiel n'est pas certifié
        partial.unlink()

        # 4. Reprise depuis la dernière frontière certifiée
        resumed = records[committed_line:]

        chunk2 = chunks / "chunk_000002.jsonl"

        with open(chunk2, "w", encoding="utf-8") as f:
            for r in resumed[:chunk_size]:
                f.write(json.dumps(r) + "\n")

        digest2 = hashlib.sha256(chunk2.read_bytes()).hexdigest()

        # 5. Vérification des segments
        valid1 = hashlib.sha256(chunk1.read_bytes()).hexdigest() == digest1
        valid2 = hashlib.sha256(chunk2.read_bytes()).hexdigest() == digest2

        ids1 = [json.loads(x)["id"] for x in chunk1.read_text().splitlines()]
        ids2 = [json.loads(x)["id"] for x in chunk2.read_text().splitlines()]

        recovered = ids1 + ids2

        return {
            "first_segment_valid": valid1,
            "second_segment_valid": valid2,
            "partial_deleted": not partial.exists(),
            "recovery_starts_at": resumed[0]["id"],
            "recovered_records": len(recovered),
            "no_duplicates": len(recovered) == len(set(recovered)),
            "continuous_sequence": recovered == list(range(1, 21)),
            "test_ok": (
                valid1 and valid2
                and not partial.exists()
                and len(recovered) == 20
                and len(recovered) == len(set(recovered))
                and recovered == list(range(1, 21))
            )
        }

result = test_interruption_v3()

print("=== TEST INTERRUPTION V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

=== TEST INTERRUPTION V3 ===
first_segment_valid : True
second_segment_valid : True
partial_deleted : True
recovery_starts_at : 11
recovered_records : 20
no_duplicates : True
continuous_sequence : True
test_ok : True
TEST : OK


In [20]:
import hashlib, json, tempfile
from pathlib import Path

def test_corruption_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_corruption_v3_") as tmp:
        root = Path(tmp)
        chunk = root / "chunk_000001.jsonl"

        records = [
            {"id": i, "title": f"Science {i}"}
            for i in range(1, 6)
        ]

        with open(chunk, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")

        original_hash = hashlib.sha256(chunk.read_bytes()).hexdigest()

        # Vérification initiale
        initial_ok = (
            hashlib.sha256(chunk.read_bytes()).hexdigest()
            == original_hash
        )

        # Simuler une modification malveillante ou accidentelle
        content = chunk.read_bytes()
        chunk.write_bytes(content.replace(b"Science 3", b"Science X"))

        corrupted_hash = hashlib.sha256(chunk.read_bytes()).hexdigest()

        corruption_detected = corrupted_hash != original_hash

        # Restaurer le fichier original
        chunk.write_bytes(content)

        restored_ok = (
            hashlib.sha256(chunk.read_bytes()).hexdigest()
            == original_hash
        )

        return {
            "initial_integrity": initial_ok,
            "corruption_detected": corruption_detected,
            "restoration_verified": restored_ok,
            "test_ok": initial_ok and corruption_detected and restored_ok
        }

result = test_corruption_v3()

print("=== TEST CORRUPTION V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

=== TEST CORRUPTION V3 ===
initial_integrity : True
corruption_detected : True
restoration_verified : True
test_ok : True
TEST : OK


In [21]:
import json, hashlib, tempfile
from pathlib import Path

def test_manifest_coherence_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_manifest_v3_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        # Création d'un segment
        chunk = chunks / "chunk_000001.jsonl"
        chunk.write_text(
            '{"id":1}\n{"id":2}\n',
            encoding="utf-8"
        )

        digest = hashlib.sha256(chunk.read_bytes()).hexdigest()

        manifest = {
            "source_id": "source_A",
            "completed_chunks": [{
                "file": chunk.name,
                "sha256": digest,
                "records": 2
            }]
        }

        # 1. Segment présent et hash valide
        entry = manifest["completed_chunks"][0]
        valid_initial = (
            chunk.exists()
            and hashlib.sha256(chunk.read_bytes()).hexdigest()
            == entry["sha256"]
        )

        # 2. Simuler corruption
        chunk.write_text('{"id":999}\n', encoding="utf-8")

        corruption_detected = (
            hashlib.sha256(chunk.read_bytes()).hexdigest()
            != entry["sha256"]
        )

        # 3. Simuler fichier absent
        chunk.unlink()
        missing_detected = not chunk.exists()

        # 4. Identité de source
        source_A_accepted = manifest["source_id"] == "source_A"
        source_B_rejected = manifest["source_id"] != "source_B"

        return {
            "initial_segment_valid": valid_initial,
            "corruption_detected": corruption_detected,
            "missing_segment_detected": missing_detected,
            "correct_source_accepted": source_A_accepted,
            "different_source_rejected": source_B_rejected,
            "test_ok": all([
                valid_initial,
                corruption_detected,
                missing_detected,
                source_A_accepted,
                source_B_rejected
            ])
        }

result = test_manifest_coherence_v3()

print("=== TEST COHÉRENCE MANIFESTE V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

=== TEST COHÉRENCE MANIFESTE V3 ===
initial_segment_valid : True
corruption_detected : True
missing_segment_detected : True
correct_source_accepted : True
different_source_rejected : True
test_ok : True
TEST : OK


In [22]:
import os, json, gzip, hashlib, time, re
from pathlib import Path
from urllib.request import Request, urlopen

# =========================================================
# NEXUS SCIENCE — SCANNER V3 (INSTALLATION SANS SCAN)
# =========================================================

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
CHUNK_SIZE = 100_000

ROOT.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

def atomic_write_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def new_manifest():
    return {
        "version": 3,
        "project": "Bibliothèque Scientifique — Nexus Science",
        "source": SOURCE,
        "source_identity": None,
        "status": "prepared",
        "scan_lines": 0,
        "valid_json": 0,
        "selected": 0,
        "invalid_json": 0,
        "out_of_period": 0,
        "completed_chunks": [],
        "updated_at_utc": None
    }

if not MANIFEST.exists():
    atomic_write_json(MANIFEST, new_manifest())

def load_manifest_v3():
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def verify_chunks_v3():
    manifest = load_manifest_v3()
    results = []

    for item in manifest["completed_chunks"]:
        path = CHUNKS / item["file"]

        if not path.exists():
            results.append((item["file"], "ABSENT"))
            continue

        actual_hash = sha256_file(path)
        if actual_hash != item["sha256"]:
            results.append((item["file"], "CORRUPTION"))
            continue

        count = 0
        invalid = 0
        with open(path, "rb") as f:
            for line in f:
                count += 1
                try:
                    json.loads(line)
                except Exception:
                    invalid += 1

        if count != item["records"] or invalid:
            results.append((item["file"], "JSONL_INVALIDE"))
        else:
            results.append((item["file"], "OK"))

    return results

def next_chunk_v3():
    manifest = load_manifest_v3()
    numbers = []

    for item in manifest["completed_chunks"]:
        match = re.search(r"chunk_(\d+)", item["file"])
        if match:
            numbers.append(int(match.group(1)))

    return max(numbers, default=0) + 1

# =========================================================
# CONTRÔLE INITIAL — AUCUN SCAN
# =========================================================

manifest = load_manifest_v3()

print("=== INSTALLATION MOTEUR NEXUS V3 ===")
print("Répertoire :", ROOT)
print("Manifeste :", MANIFEST)
print("Checkpoint :", CHECKPOINT)
print("Segments enregistrés :", len(manifest["completed_chunks"]))
print("Prochain segment :", next_chunk_v3())
print("Taille segment :", CHUNK_SIZE)
print("Scan massif : NON")
print("Installation : OK")

=== INSTALLATION MOTEUR NEXUS V3 ===
Répertoire : /kaggle/working/nexus_science_v3
Manifeste : /kaggle/working/nexus_science_v3/manifest_v3.json
Checkpoint : /kaggle/working/nexus_science_v3/checkpoint_v3.json
Segments enregistrés : 0
Prochain segment : 1
Taille segment : 100000
Scan massif : NON
Installation : OK


In [23]:
import os, json, gzip, hashlib, re, time
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

CHUNK_SIZE = 100_000
SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

# Filtrage configurable
SCIENCE_KEYWORDS = [
    "science", "scientific", "physics", "mathematics", "math",
    "chemistry", "biology", "astronomy", "astrophysics",
    "engineering", "medicine", "medical", "geology",
    "statistics", "computer science", "algorithm",
    "research", "technology", "ecology", "genetics",
    "neuroscience", "quantum", "anatomy", "botany",
    "zoology", "microbiology", "biochemistry", "thermodynamics",
    "électronique", "physique", "mathématique", "chimie",
    "biologie", "médecine", "astronomie", "géologie",
    "scientifique", "recherche", "technologie", "ingénierie",
    "estatística", "matemática", "ciência", "fisica",
    "química", "biologia", "medicina", "engenharia"
]

EXCLUSIONS = [
    "fiction", "romance", "poetry", "comic", "comics",
    "novel", "fantasy", "thriller"
]

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def scientific_record(record):
    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])
    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    if any(word in text for word in EXCLUSIONS):
        return False

    return any(word in text for word in SCIENCE_KEYWORDS)

def extract_year(record):
    for key in ("publish_date", "date"):
        value = record.get(key)
        if value is not None:
            match = re.search(r"\b(1[7-9]\d{2}|20\d{2})\b", str(value))
            if match:
                return int(match.group(1))
    return None

def in_period(record):
    year = extract_year(record)
    return year is not None and 1720 <= year <= 2026

def validate_jsonl(path, expected_count):
    count = 0
    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1
    return count == expected_count

def certify_chunk(path, expected_count):
    if not validate_jsonl(path, expected_count):
        raise RuntimeError("Validation JSONL échouée")

    return {
        "records": expected_count,
        "bytes": path.stat().st_size,
        "sha256": hash_file(path)
    }

def committed_source_line(manifest):
    if not manifest["completed_chunks"]:
        return 0
    return manifest["completed_chunks"][-1]["source_line_end"]

def verify_manifest_chunks():
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

    for item in manifest["completed_chunks"]:
        path = CHUNKS / item["file"]

        if not path.exists():
            raise RuntimeError(f"Segment absent : {item['file']}")

        if hash_file(path) != item["sha256"]:
            raise RuntimeError(f"Hash incorrect : {item['file']}")

        if not validate_jsonl(path, item["records"]):
            raise RuntimeError(f"JSONL invalide : {item['file']}")

    return manifest

class NexusV3Engine:
    def __init__(self):
        self.manifest = verify_manifest_chunks()
        self.next_number = max(
            [
                int(re.search(r"chunk_(\d+)", x["file"]).group(1))
                for x in self.manifest["completed_chunks"]
            ],
            default=0
        ) + 1

    def commit_segment(self, records, source_line_end, counters):
        if not records:
            return None

        final_name = f"chunk_{self.next_number:06d}.jsonl"
        final_path = CHUNKS / final_name
        temp_path = CHUNKS / (final_name + ".partial")

        # Écriture non certifiée
        with open(temp_path, "wb") as f:
            for record in records:
                f.write(
                    json.dumps(record, ensure_ascii=False).encode("utf-8")
                    + b"\n"
                )
            f.flush()
            os.fsync(f.fileno())

        # Validation avant certification
        info = certify_chunk(temp_path, len(records))

        # Renommage du segment validé
        os.replace(temp_path, final_path)

        entry = {
            "file": final_name,
            "records": info["records"],
            "bytes": info["bytes"],
            "sha256": info["sha256"],
            "source_line_end": source_line_end,
            "committed_at_utc": utc_now()
        }

        # Manifeste mis à jour uniquement après certification
        self.manifest["completed_chunks"].append(entry)
        self.manifest["scan_lines"] = source_line_end
        self.manifest["selected"] += len(records)
        self.manifest["updated_at_utc"] = utc_now()
        self.manifest["status"] = "running"

        atomic_json(MANIFEST, self.manifest)

        # Checkpoint dérivé du manifeste certifié
        checkpoint = {
            "version": 3,
            "source": SOURCE,
            "scan_lines": source_line_end,
            "selected": self.manifest["selected"],
            "chunk_number": self.next_number,
            "source_identity": self.manifest.get("source_identity"),
            "completed_chunks": self.manifest["completed_chunks"]
        }

        atomic_json(CHECKPOINT, checkpoint)

        self.next_number += 1
        return entry

print("=== MOTEUR DE TRAITEMENT V3 INSTALLÉ ===")
print("Filtrage scientifique : prêt")
print("Validation JSONL : prête")
print("Certification SHA-256 : prête")
print("Checkpoint post-certification : prêt")
print("Reprise par frontière certifiée : préparée")
print("Scan massif : NON")

=== MOTEUR DE TRAITEMENT V3 INSTALLÉ ===
Filtrage scientifique : prêt
Validation JSONL : prête
Certification SHA-256 : prête
Checkpoint post-certification : prêt
Reprise par frontière certifiée : préparée
Scan massif : NON


In [24]:
import json, hashlib, os, tempfile
from pathlib import Path

def test_transaction_v3():
    with tempfile.TemporaryDirectory(prefix="nexus_tx_v3_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        manifest_path = root / "manifest.json"
        checkpoint_path = root / "checkpoint.json"

        source_identity = {
            "url": "mini_source.gz",
            "sha256": "SIMULATION_SOURCE_123"
        }

        manifest = {
            "version": 3,
            "source_identity": source_identity,
            "scan_lines": 0,
            "selected": 0,
            "completed_chunks": []
        }

        records = [{"id": i} for i in range(1, 11)]
        source_line_end = 1000

        # Écriture temporaire
        partial = chunks / "chunk_000001.jsonl.partial"
        final = chunks / "chunk_000001.jsonl"

        with open(partial, "wb") as f:
            for r in records:
                f.write(json.dumps(r).encode() + b"\n")
            f.flush()
            os.fsync(f.fileno())

        # Validation JSONL
        parsed = []
        with open(partial, "rb") as f:
            for line in f:
                parsed.append(json.loads(line))

        valid_jsonl = len(parsed) == 10

        digest = hashlib.sha256(partial.read_bytes()).hexdigest()
        size = partial.stat().st_size

        # Certification
        os.replace(partial, final)

        entry = {
            "file": final.name,
            "records": len(parsed),
            "bytes": size,
            "sha256": digest,
            "source_line_end": source_line_end
        }

        manifest["completed_chunks"].append(entry)
        manifest["scan_lines"] = source_line_end
        manifest["selected"] = len(parsed)

        # Manifeste atomique
        tmp_manifest = root / "manifest.tmp"
        tmp_manifest.write_text(
            json.dumps(manifest, indent=2),
            encoding="utf-8"
        )
        os.replace(tmp_manifest, manifest_path)

        # Checkpoint dérivé du manifeste
        checkpoint = {
            "version": 3,
            "source_identity": source_identity,
            "scan_lines": manifest["scan_lines"],
            "selected": manifest["selected"],
            "completed_chunks": manifest["completed_chunks"]
        }

        tmp_checkpoint = root / "checkpoint.tmp"
        tmp_checkpoint.write_text(
            json.dumps(checkpoint, indent=2),
            encoding="utf-8"
        )
        os.replace(tmp_checkpoint, checkpoint_path)

        # Relecture
        recovered_manifest = json.loads(
            manifest_path.read_text(encoding="utf-8")
        )
        recovered_checkpoint = json.loads(
            checkpoint_path.read_text(encoding="utf-8")
        )

        recovered_file = chunks / entry["file"]
        recovered_hash = hashlib.sha256(
            recovered_file.read_bytes()
        ).hexdigest()

        source_match = (
            recovered_manifest["source_identity"]
            == source_identity
            == recovered_checkpoint["source_identity"]
        )

        boundary_match = (
            recovered_manifest["scan_lines"]
            == recovered_checkpoint["scan_lines"]
            == source_line_end
        )

        hash_match = recovered_hash == entry["sha256"]

        records_match = (
            recovered_manifest["selected"]
            == recovered_checkpoint["selected"]
            == 10
        )

        return {
            "jsonl_valid": valid_jsonl,
            "segment_certified": hash_match,
            "source_identity_match": source_match,
            "boundary_match": boundary_match,
            "records_match": records_match,
            "checkpoint_recovered": checkpoint_path.exists(),
            "test_ok": all([
                valid_jsonl,
                hash_match,
                source_match,
                boundary_match,
                records_match,
                checkpoint_path.exists()
            ])
        }

result = test_transaction_v3()

print("=== TEST TRANSACTION INTÉGRÉE V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

=== TEST TRANSACTION INTÉGRÉE V3 ===
jsonl_valid : True
segment_certified : True
source_identity_match : True
boundary_match : True
records_match : True
checkpoint_recovered : True
test_ok : True
TEST : OK


In [25]:
import json, hashlib, tempfile, os
from pathlib import Path

def test_crash_windows_v3():
    results = {}

    with tempfile.TemporaryDirectory(prefix="nexus_crash_windows_") as tmp:
        root = Path(tmp)
        chunks = root / "chunks"
        chunks.mkdir()

        manifest_path = root / "manifest.json"
        checkpoint_path = root / "checkpoint.json"

        def save_atomic(path, data):
            temp = path.with_suffix(".tmp")
            temp.write_text(json.dumps(data), encoding="utf-8")
            os.replace(temp, path)

        initial = {
            "scan_lines": 0,
            "selected": 0,
            "completed_chunks": []
        }
        save_atomic(manifest_path, initial)
        save_atomic(checkpoint_path, initial)

        # -----------------------------------------------
        # SCÉNARIO 1 : segment renommé, manifeste absent
        # -----------------------------------------------
        orphan = chunks / "chunk_000001.jsonl"
        orphan.write_text('{"id":1}\n{"id":2}\n', encoding="utf-8")

        orphan_hash = hashlib.sha256(orphan.read_bytes()).hexdigest()
        orphan_valid = orphan_hash == hashlib.sha256(
            orphan.read_bytes()
        ).hexdigest()

        manifest1 = json.loads(manifest_path.read_text())
        orphan_not_registered = (
            len(manifest1["completed_chunks"]) == 0
        )

        results["orphan_detected_as_unregistered"] = (
            orphan.exists() and orphan_not_registered and orphan_valid
        )

        # Ne pas adopter automatiquement l'orphelin.
        orphan.unlink()

        # -----------------------------------------------
        # SCÉNARIO 2 : manifeste avancé, checkpoint ancien
        # -----------------------------------------------
        chunk2 = chunks / "chunk_000001.jsonl"
        chunk2.write_text('{"id":1}\n{"id":2}\n', encoding="utf-8")

        digest2 = hashlib.sha256(chunk2.read_bytes()).hexdigest()

        committed = {
            "scan_lines": 500,
            "selected": 2,
            "completed_chunks": [{
                "file": chunk2.name,
                "records": 2,
                "sha256": digest2,
                "source_line_end": 500
            }]
        }

        save_atomic(manifest_path, committed)

        # Le checkpoint reste volontairement à zéro.
        stale_checkpoint = {
            "scan_lines": 0,
            "selected": 0,
            "completed_chunks": []
        }
        save_atomic(checkpoint_path, stale_checkpoint)

        # Le manifeste est la référence maîtresse.
        recovered = json.loads(manifest_path.read_text())
        reconstructed = {
            "scan_lines": recovered["scan_lines"],
            "selected": recovered["selected"],
            "completed_chunks": recovered["completed_chunks"]
        }

        results["manifest_authoritative"] = (
            reconstructed["scan_lines"] == 500
            and reconstructed["selected"] == 2
        )

        results["stale_checkpoint_reconstructable"] = (
            reconstructed == committed
        )

        # -----------------------------------------------
        # SCÉNARIO 3 : segment certifié corrompu
        # -----------------------------------------------
        chunk2.write_text('{"id":999}\n', encoding="utf-8")

        corruption_detected = (
            hashlib.sha256(chunk2.read_bytes()).hexdigest()
            != committed["completed_chunks"][0]["sha256"]
        )

        results["corruption_blocks_recovery"] = corruption_detected

        results["test_ok"] = all([
            results["orphan_detected_as_unregistered"],
            results["manifest_authoritative"],
            results["stale_checkpoint_reconstructable"],
            results["corruption_blocks_recovery"]
        ])

    return results

result = test_crash_windows_v3()

print("=== TEST FENÊTRES DE PANNE V3 ===")
for key, value in result.items():
    print(f"{key} : {value}")

print("TEST :", "OK" if result["test_ok"] else "ÉCHEC")

=== TEST FENÊTRES DE PANNE V3 ===
orphan_detected_as_unregistered : True
manifest_authoritative : True
stale_checkpoint_reconstructable : True
corruption_blocks_recovery : True
test_ok : True
TEST : OK


In [26]:
import json, hashlib, os, re
from pathlib import Path

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

def v3_atomic_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

def v3_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_validate_jsonl(path, expected):
    count = 0
    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1
    return count == expected

def v3_recovery_controller():
    manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    registered = set()
    verified = 0

    # Vérifier tous les segments enregistrés
    for item in manifest["completed_chunks"]:
        filename = item["file"]
        registered.add(filename)
        path = CHUNKS / filename

        if not path.exists():
            raise RuntimeError(f"SEGMENT ABSENT : {filename}")

        if v3_hash(path) != item["sha256"]:
            raise RuntimeError(f"CORRUPTION HASH : {filename}")

        if not v3_validate_jsonl(path, item["records"]):
            raise RuntimeError(f"JSONL INVALIDE : {filename}")

        if "source_line_end" not in item:
            raise RuntimeError(
                f"FRONTIÈRE SOURCE ABSENTE : {filename}"
            )

        verified += 1

    # Repérer les fichiers non enregistrés
    orphans = []
    for path in CHUNKS.glob("chunk_*.jsonl"):
        if path.name not in registered:
            orphans.append(path.name)

    # Ne jamais adopter automatiquement les orphelins
    if orphans:
        return {
            "status": "ORPHANS_BLOCK_RECOVERY",
            "verified": verified,
            "orphans": sorted(orphans),
            "checkpoint_rebuilt": False
        }

    # Reconstruction du checkpoint depuis le manifeste
    checkpoint = {
        "version": 3,
        "source": manifest.get("source"),
        "source_identity": manifest.get("source_identity"),
        "scan_lines": manifest["scan_lines"],
        "selected": manifest["selected"],
        "completed_chunks": manifest["completed_chunks"]
    }

    v3_atomic_json(CHECKPOINT, checkpoint)

    return {
        "status": "RECOVERY_READY",
        "verified": verified,
        "orphans": [],
        "checkpoint_rebuilt": True,
        "scan_lines": manifest["scan_lines"],
        "selected": manifest["selected"]
    }

print("=== CONTRÔLEUR DE RÉCUPÉRATION V3 ===")
result = v3_recovery_controller()

for key, value in result.items():
    print(f"{key} : {value}")

print("SCAN MASSIF : NON")

=== CONTRÔLEUR DE RÉCUPÉRATION V3 ===
status : RECOVERY_READY
verified : 0
orphans : []
checkpoint_rebuilt : True
scan_lines : 0
selected : 0
SCAN MASSIF : NON


In [29]:
import os, json, gzip, hashlib, re, time
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen

# ============================================================
# NEXUS SCIENCE — SCANNER V3 INTÉGRÉ
# INSTALLATION SEULE : AUCUN SCAN
# ============================================================

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

CHUNK_SIZE = 100_000
MIN_YEAR = 1720
MAX_YEAR = 2026

RUN_SCAN = False   # VERROU DE SÉCURITÉ

ROOT.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# UTILITAIRES
# ------------------------------------------------------------

def v3_now():
    return datetime.now(timezone.utc).isoformat()

def v3_atomic_json(path, data):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")

    with open(temp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, path)

def v3_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_manifest():
    if not MANIFEST.exists():
        raise RuntimeError("Manifeste absent")
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def v3_validate(path, expected_records):
    count = 0

    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1

    return count == expected_records

# ------------------------------------------------------------
# IDENTITÉ SOURCE
# ------------------------------------------------------------

def v3_source_identity():
    req = Request(
        SOURCE,
        headers={
            "User-Agent": "NexusScienceScanner/3.0",
            "Accept-Encoding": "identity"
        },
        method="HEAD"
    )

    with urlopen(req, timeout=30) as response:
        headers = response.headers

        return {
            "url": SOURCE,
            "etag": headers.get("ETag"),
            "last_modified": headers.get("Last-Modified"),
            "content_length": headers.get("Content-Length")
        }

# ------------------------------------------------------------
# FILTRAGE
# ------------------------------------------------------------

SCIENCE_KEYWORDS = [
    "science", "scientific", "physics", "mathematics",
    "math", "chemistry", "biology", "astronomy",
    "astrophysics", "engineering", "medicine",
    "medical", "geology", "statistics", "computer science",
    "algorithm", "research", "technology", "ecology",
    "genetics", "neuroscience", "quantum", "anatomy",
    "botany", "zoology", "microbiology", "biochemistry",
    "thermodynamics", "électronique", "physique",
    "mathématique", "chimie", "biologie", "médecine",
    "astronomie", "géologie", "scientifique", "recherche",
    "technologie", "ingénierie", "matemática", "ciência",
    "química", "medicina", "engenharia"
]

EXCLUSIONS = [
    "romance", "poetry", "comic", "comics",
    "thriller", "fantasy"
]

def v3_is_scientific(record):
    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])

    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    if any(term in text for term in EXCLUSIONS):
        return False

    return any(term in text for term in SCIENCE_KEYWORDS)

def v3_year(record):
    for key in ("publish_date", "date"):
        value = record.get(key)

        if value is not None:
            match = re.search(
                r"\b(1[7-9]\d{2}|20\d{2})\b",
                str(value)
            )
            if match:
                return int(match.group(1))

    return None

def v3_in_period(record):
    year = v3_year(record)
    return year is not None and MIN_YEAR <= year <= MAX_YEAR

# ------------------------------------------------------------
# VÉRIFICATION DES SEGMENTS ENREGISTRÉS
# ------------------------------------------------------------

def v3_verify_registered(manifest):
    registered = set()

    for item in manifest["completed_chunks"]:
        name = item["file"]
        registered.add(name)
        path = CHUNKS / name

        if not path.exists():
            raise RuntimeError("SEGMENT ABSENT : " + name)

        if v3_sha256(path) != item["sha256"]:
            raise RuntimeError("HASH INCORRECT : " + name)

        if not v3_validate(path, item["records"]):
            raise RuntimeError("JSONL INVALIDE : " + name)

        if "source_line_end" not in item:
            raise RuntimeError(
                "FRONTIÈRE SOURCE MANQUANTE : " + name
            )

    # Orphelins : refus de démarrer sans intervention
    orphans = [
        p.name for p in CHUNKS.glob("chunk_*.jsonl")
        if p.name not in registered
    ]

    partials = [
        p.name for p in CHUNKS.glob("*.partial")
    ]

    if orphans or partials:
        raise RuntimeError(
            f"Récupération bloquée : orphelins={orphans}, "
            f"partiels={partials}"
        )

# ------------------------------------------------------------
# MOTEUR
# ------------------------------------------------------------

class NexusScannerV3:

    def __init__(self):
        self.manifest = v3_manifest()
        v3_verify_registered(self.manifest)

        self.next_number = max(
            [
                int(re.search(r"chunk_(\d+)", x["file"]).group(1))
                for x in self.manifest["completed_chunks"]
            ],
            default=0
        ) + 1

        self.identity = self.manifest.get("source_identity")
        self.resume_line = self.manifest.get("scan_lines", 0)

    def verify_source(self, identity):
        previous = self.manifest.get("source_identity")

        if previous is not None and previous != identity:
            raise RuntimeError(
                "SOURCE DIFFÉRENTE : reprise refusée"
            )

        if previous is None and self.resume_line != 0:
            raise RuntimeError(
                "Checkpoint sans identité source : reprise refusée"
            )

        self.identity = identity

    def commit(self, records, source_line_end, counters):
        if not records:
            # Une frontière sans segment ne sera pas commitée
            return

        name = f"chunk_{self.next_number:06d}.jsonl"
        temp = CHUNKS / (name + ".partial")
        final = CHUNKS / name

        # Écriture du segment temporaire
        with open(temp, "wb") as f:
            for record in records:
                f.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    ).encode("utf-8") + b"\n"
                )

            f.flush()
            os.fsync(f.fileno())

        # Validation avant certification
        if not v3_validate(temp, len(records)):
            raise RuntimeError("Validation JSONL échouée")

        digest = v3_sha256(temp)
        size = temp.stat().st_size

        # Certification et renommage
        os.replace(temp, final)

        entry = {
            "file": name,
            "records": len(records),
            "bytes": size,
            "sha256": digest,
            "source_line_end": source_line_end,
            "committed_at_utc": v3_now()
        }

        # Le manifeste devient la référence maîtresse
        updated = dict(self.manifest)
        updated["completed_chunks"] = (
            self.manifest["completed_chunks"] + [entry]
        )

        updated["scan_lines"] = source_line_end
        updated["selected"] = (
            self.manifest["selected"] + len(records)
        )

        for key, value in counters.items():
            updated[key] = self.manifest.get(key, 0) + value

        updated["source_identity"] = self.identity
        updated["status"] = "running"
        updated["updated_at_utc"] = v3_now()

        v3_atomic_json(MANIFEST, updated)

        # Checkpoint dérivé du manifeste
        checkpoint = {
            "version": 3,
            "source": SOURCE,
            "source_identity": updated["source_identity"],
            "scan_lines": updated["scan_lines"],
            "selected": updated["selected"],
            "completed_chunks": updated["completed_chunks"]
        }

        v3_atomic_json(CHECKPOINT, checkpoint)

        self.manifest = updated
        self.next_number += 1

    def scan(self):
        raise NotImplementedError(
            "Le flux réel nécessite le raccordement final "
            "et son test avant activation."
        )

# ------------------------------------------------------------
# INSTALLATION ET CONTRÔLE
# ------------------------------------------------------------

scanner = NexusScannerV3()

print("=== NEXUS SCIENCE — SCANNER V3 ===")
print("Moteur intégré : installé")
print("Source : configurée")
print("Filtrage : configuré")
print("Certification : configurée")
print("Gestionnaire de récupération : raccordé")
print("Prochain segment :", scanner.next_number)
print("Frontière de reprise :", scanner.resume_line)
print("RUN_SCAN :", RUN_SCAN)
print("SCAN MASSIF : NON")
print("INSTALLATION : OK")

=== NEXUS SCIENCE — SCANNER V3 ===
Moteur intégré : installé
Source : configurée
Filtrage : configuré
Certification : configurée
Gestionnaire de récupération : raccordé
Prochain segment : 1
Frontière de reprise : 0
RUN_SCAN : False
SCAN MASSIF : NON
INSTALLATION : OK


In [31]:
import gzip, json, hashlib, time, os
from pathlib import Path
from urllib.request import Request, urlopen

# =========================================================
# NEXUS V3 — TEST LIMITÉ DU FLUX OPEN LIBRARY
# =========================================================

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"
LIMIT_LINES = 20_000

TEST_ROOT = Path("/kaggle/working/nexus_science_v3/integration_test")
TEST_ROOT.mkdir(parents=True, exist_ok=True)

TEST_CHUNK = TEST_ROOT / "test_chunk.jsonl"

def test_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def test_scientific_filter(record):
    keywords = [
        "science", "scientific", "physics", "mathematics",
        "math", "chemistry", "biology", "astronomy",
        "engineering", "medicine", "medical", "geology",
        "statistics", "computer science", "research",
        "technology", "ecology", "genetics", "neuroscience",
        "quantum", "anatomy", "botany", "zoology",
        "microbiology", "biochemistry", "thermodynamics",
        "physique", "mathématique", "chimie", "biologie",
        "médecine", "astronomie", "géologie", "scientifique",
        "recherche", "technologie", "ingénierie",
        "matemática", "ciência", "química", "medicina",
        "engenharia"
    ]

    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])

    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    return any(k in text for k in keywords)

def test_period_filter(record):
    value = record.get("publish_date", record.get("date", ""))

    import re
    match = re.search(
        r"\b(1[7-9]\d{2}|20\d{2})\b",
        str(value)
    )

    if not match:
        return False

    year = int(match.group(1))
    return 1720 <= year <= 2026

# ---------------------------------------------------------
# 1. IDENTITÉ SOURCE
# ---------------------------------------------------------

identity = None

try:
    req = Request(
        SOURCE,
        method="HEAD",
        headers={"User-Agent": "NexusScienceScanner/3.0"}
    )

    with urlopen(req, timeout=30) as response:
        identity = {
            "url": SOURCE,
            "etag": response.headers.get("ETag"),
            "last_modified": response.headers.get("Last-Modified"),
            "content_length": response.headers.get("Content-Length")
        }

    print("IDENTITÉ SOURCE : OK")
    print("ETag :", identity["etag"])
    print("Last-Modified :", identity["last_modified"])
    print("Content-Length :", identity["content_length"])

except Exception as e:
    raise RuntimeError(
        f"Échec de vérification de la source : {e}"
    )

# ---------------------------------------------------------
# 2. LECTURE LIMITÉE
# ---------------------------------------------------------

physical_lines = 0
valid_json = 0
invalid_json = 0
in_period = 0
selected = 0

records_buffer = []
start_time = time.time()

try:
    req = Request(
        SOURCE,
        headers={"User-Agent": "NexusScienceScanner/3.0"}
    )

    with urlopen(req, timeout=60) as response:
        with gzip.GzipFile(fileobj=response) as gz:

            while physical_lines < LIMIT_LINES:
                raw = gz.readline()

                if not raw:
                    break

                physical_lines += 1

                try:
                    parts = raw.rstrip(b"\r\n").split(b"\t")
                    record = json.loads(parts[-1])

                    if not isinstance(record, dict):
                        invalid_json += 1
                        continue

                    valid_json += 1

                except Exception:
                    invalid_json += 1
                    continue

                if not test_period_filter(record):
                    continue

                in_period += 1

                if test_scientific_filter(record):
                    records_buffer.append(record)
                    selected += 1

except Exception as e:
    raise RuntimeError(f"Échec du flux limité : {e}")

# ---------------------------------------------------------
# 3. ÉCRITURE DU SEGMENT DE TEST
# ---------------------------------------------------------

with open(TEST_CHUNK, "wb") as f:
    for record in records_buffer:
        f.write(
            json.dumps(record, ensure_ascii=False).encode("utf-8")
            + b"\n"
        )

    f.flush()
    os.fsync(f.fileno())

# ---------------------------------------------------------
# 4. VALIDATION
# ---------------------------------------------------------

validated = 0
jsonl_errors = 0

with open(TEST_CHUNK, "rb") as f:
    for line in f:
        try:
            json.loads(line)
            validated += 1
        except Exception:
            jsonl_errors += 1

digest = test_hash(TEST_CHUNK)
size_bytes = TEST_CHUNK.stat().st_size

test_ok = (
    physical_lines > 0
    and valid_json + invalid_json == physical_lines
    and validated == selected
    and jsonl_errors == 0
)

print()
print("=== TEST FLUX RÉEL V3 ===")
print("Lignes physiques :", physical_lines)
print("JSON valides :", valid_json)
print("JSON invalides :", invalid_json)
print("Dans la période :", in_period)
print("Notices sélectionnées :", selected)
print("Notices validées :", validated)
print("Erreurs JSONL :", jsonl_errors)
print("Taille segment (octets) :", size_bytes)
print("SHA-256 :", digest)
print("Temps (secondes) :", round(time.time() - start_time, 2))
print("TEST :", "OK" if test_ok else "ÉCHEC")
print("SCAN MASSIF : NON")

IDENTITÉ SOURCE : OK
ETag : "6ae8a85f-2f0dac92a"
Last-Modified : Mon, 02 Nov 2026 15:12:31 GMT
Content-Length : 12630804778

=== TEST FLUX RÉEL V3 ===
Lignes physiques : 20000
JSON valides : 20000
JSON invalides : 0
Dans la période : 19290
Notices sélectionnées : 2093
Notices validées : 2093
Erreurs JSONL : 0
Taille segment (octets) : 2199659
SHA-256 : 33777a1ea0f7fe37917edf90865f4fd83d7dd6528d45c9fe6faf011751ee9ce6
Temps (secondes) : 2.1
TEST : OK
SCAN MASSIF : NON


In [33]:
import os, json, gzip, hashlib, re, time
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen

# ============================================================
# NEXUS SCIENCE V3 — MOTEUR SEGMENTÉ ET REPRENABLE
# ============================================================

ROOT = Path("/kaggle/working/nexus_science_v3")
CHUNKS = ROOT / "chunks"
MANIFEST = ROOT / "manifest_v3.json"
CHECKPOINT = ROOT / "checkpoint_v3.json"

SOURCE = "https://openlibrary.org/data/ol_dump_editions_latest.txt.gz"

CHUNK_SIZE = 100_000
MIN_YEAR = 1720
MAX_YEAR = 2026

RUN_SCAN = False

ROOT.mkdir(parents=True, exist_ok=True)
CHUNKS.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# UTILITAIRES
# ------------------------------------------------------------

def now_utc():
    return datetime.now(timezone.utc).isoformat()

def atomic_json(path, data):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def load_manifest():
    if not MANIFEST.exists():
        raise RuntimeError("Manifeste absent")
    return json.loads(MANIFEST.read_text(encoding="utf-8"))

def source_identity():
    req = Request(
        SOURCE,
        method="HEAD",
        headers={"User-Agent": "NexusScienceScanner/3.0"}
    )

    with urlopen(req, timeout=30) as response:
        return {
            "url": SOURCE,
            "etag": response.headers.get("ETag"),
            "last_modified": response.headers.get("Last-Modified"),
            "content_length": response.headers.get("Content-Length")
        }

# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

def validate_jsonl(path, expected):
    count = 0

    with open(path, "rb") as f:
        for line in f:
            if not line.strip():
                return False
            try:
                json.loads(line)
            except Exception:
                return False
            count += 1

    return count == expected

def verify_manifest_segments(manifest):
    registered = set()

    for entry in manifest["completed_chunks"]:
        filename = entry["file"]
        registered.add(filename)
        path = CHUNKS / filename

        if not path.exists():
            raise RuntimeError(f"SEGMENT ABSENT : {filename}")

        if sha256_file(path) != entry["sha256"]:
            raise RuntimeError(f"HASH INCORRECT : {filename}")

        if not validate_jsonl(path, entry["records"]):
            raise RuntimeError(f"JSONL INVALIDE : {filename}")

        if "source_line_end" not in entry:
            raise RuntimeError(
                f"FRONTIÈRE SOURCE MANQUANTE : {filename}"
            )

    # Refus d'adopter automatiquement les segments orphelins
    orphans = [
        p.name for p in CHUNKS.glob("chunk_*.jsonl")
        if p.name not in registered
    ]

    partials = list(CHUNKS.glob("*.partial"))

    if orphans or partials:
        raise RuntimeError(
            f"Récupération bloquée : orphelins={orphans}, "
            f"partiels={[p.name for p in partials]}"
        )

# ------------------------------------------------------------
# FILTRAGE
# ------------------------------------------------------------

KEYWORDS = [
    "science", "scientific", "physics", "mathematics",
    "math", "chemistry", "biology", "astronomy",
    "astrophysics", "engineering", "medicine", "medical",
    "geology", "statistics", "computer science", "algorithm",
    "research", "technology", "ecology", "genetics",
    "neuroscience", "quantum", "anatomy", "botany",
    "zoology", "microbiology", "biochemistry",
    "thermodynamics", "physique", "mathématique", "chimie",
    "biologie", "médecine", "astronomie", "géologie",
    "scientifique", "recherche", "technologie", "ingénierie",
    "matemática", "ciência", "química", "medicina",
    "engenharia"
]

EXCLUSIONS = [
    "romance", "poetry", "comic", "comics", "thriller"
]

def is_scientific(record):
    title = str(record.get("title", ""))
    subjects = record.get("subjects", [])

    if not isinstance(subjects, list):
        subjects = []

    text = (title + " " + " ".join(map(str, subjects))).lower()

    if any(word in text for word in EXCLUSIONS):
        return False

    return any(word in text for word in KEYWORDS)

def extract_year(record):
    for key in ("publish_date", "date"):
        value = record.get(key)

        if value is not None:
            match = re.search(
                r"\b(1[7-9]\d{2}|20\d{2})\b",
                str(value)
            )
            if match:
                return int(match.group(1))

    return None

def in_period(record):
    year = extract_year(record)
    return year is not None and MIN_YEAR <= year <= MAX_YEAR

# ------------------------------------------------------------
# MANIFESTE ET REPRISE
# ------------------------------------------------------------

def initialize_manifest_if_needed():
    if not MANIFEST.exists():
        atomic_json(MANIFEST, {
            "version": 3,
            "project": "Bibliothèque Scientifique — Nexus Science",
            "source": SOURCE,
            "source_identity": None,
            "status": "prepared",
            "scan_lines": 0,
            "valid_json": 0,
            "selected": 0,
            "invalid_json": 0,
            "out_of_period": 0,
            "completed_chunks": [],
            "updated_at_utc": None
        })

initialize_manifest_if_needed()

class NexusScannerV3:

    def __init__(self):
        self.manifest = load_manifest()
        verify_manifest_segments(self.manifest)

        self.next_number = max(
            [
                int(re.search(
                    r"chunk_(\d+)",
                    entry["file"]
                ).group(1))
                for entry in self.manifest["completed_chunks"]
            ],
            default=0
        ) + 1

        self.committed_line = self.manifest["scan_lines"]
        self.buffer = []
        self.interval_counters = {
            "valid_json": 0,
            "invalid_json": 0,
            "out_of_period": 0
        }

    def verify_source(self, identity):
        previous = self.manifest.get("source_identity")

        if previous is not None and previous != identity:
            raise RuntimeError(
                "Identité de source différente : reprise refusée."
            )

        if previous is None and self.committed_line != 0:
            raise RuntimeError(
                "Checkpoint sans identité source : reprise refusée."
            )

        self.identity = identity

    def commit(self, source_line_end):
        if not self.buffer:
            return

        name = f"chunk_{self.next_number:06d}.jsonl"
        temp = CHUNKS / (name + ".partial")
        final = CHUNKS / name

        if final.exists():
            raise RuntimeError(
                "Refus d'écraser un segment existant."
            )

        # Écriture temporaire
        with open(temp, "wb") as f:
            for record in self.buffer:
                f.write(
                    json.dumps(
                        record, ensure_ascii=False
                    ).encode("utf-8") + b"\n"
                )

            f.flush()
            os.fsync(f.fileno())

        # Validation
        if not validate_jsonl(temp, len(self.buffer)):
            raise RuntimeError("Validation du segment échouée")

        digest = sha256_file(temp)
        size = temp.stat().st_size

        # Certification : renommage
        os.replace(temp, final)

        entry = {
            "file": name,
            "records": len(self.buffer),
            "bytes": size,
            "sha256": digest,
            "source_line_end": source_line_end,
            "committed_at_utc": now_utc()
        }

        # Manifeste maître
        updated = dict(self.manifest)
        updated["completed_chunks"] = (
            self.manifest["completed_chunks"] + [entry]
        )
        updated["scan_lines"] = source_line_end
        updated["selected"] = (
            self.manifest["selected"] + len(self.buffer)
        )

        for key, value in self.interval_counters.items():
            updated[key] = self.manifest.get(key, 0) + value

        updated["source_identity"] = self.identity
        updated["status"] = "running"
        updated["updated_at_utc"] = now_utc()

        atomic_json(MANIFEST, updated)

        # Checkpoint secondaire
        checkpoint = {
            "version": 3,
            "source": SOURCE,
            "source_identity": updated["source_identity"],
            "scan_lines": updated["scan_lines"],
            "selected": updated["selected"],
            "completed_chunks": updated["completed_chunks"]
        }

        atomic_json(CHECKPOINT, checkpoint)

        # Mise à jour mémoire
        self.manifest = updated
        self.committed_line = source_line_end
        self.next_number += 1
        self.buffer = []
        self.interval_counters = {
            "valid_json": 0,
            "invalid_json": 0,
            "out_of_period": 0
        }

    def scan_limited(self, max_lines=None):
        identity = source_identity()
        self.verify_source(identity)

        physical_lines = 0
        start_time = time.time()

        req = Request(
            SOURCE,
            headers={"User-Agent": "NexusScienceScanner/3.0"}
        )

        with urlopen(req, timeout=60) as response:
            remote_identity = {
                "url": SOURCE,
                "etag": response.headers.get("ETag"),
                "last_modified": response.headers.get("Last-Modified"),
                "content_length": response.headers.get("Content-Length")
            }

            if remote_identity != identity:
                raise RuntimeError(
                    "Identité incohérente entre HEAD et GET."
                )

            with gzip.GzipFile(fileobj=response) as gz:

                # Rejouer le flux jusqu'à la frontière certifiée
                while physical_lines < self.committed_line:
                    raw = gz.readline()
                    if not raw:
                        raise RuntimeError(
                            "Source trop courte pour le checkpoint."
                        )
                    physical_lines += 1

                while True:
                    if max_lines is not None:
                        if physical_lines - self.committed_line >= max_lines:
                            break

                    raw = gz.readline()
                    if not raw:
                        break

                    physical_lines += 1

                    try:
                        parts = raw.rstrip(b"\r\n").split(b"\t")
                        record = json.loads(parts[-1])

                        if not isinstance(record, dict):
                            raise ValueError("JSON non objet")

                        self.interval_counters["valid_json"] += 1

                    except Exception:
                        self.interval_counters["invalid_json"] += 1
                        continue

                    if not in_period(record):
                        self.interval_counters["out_of_period"] += 1
                        continue

                    if is_scientific(record):
                        self.buffer.append(record)

                        if len(self.buffer) >= CHUNK_SIZE:
                            self.commit(physical_lines)

                # Segment final du test limité
                if self.buffer:
                    self.commit(physical_lines)

        return {
            "physical_lines": physical_lines,
            "committed_line": self.committed_line,
            "selected_total": self.manifest["selected"],
            "elapsed_seconds": round(time.time() - start_time, 2)
        }

# ------------------------------------------------------------
# INSTALLATION — AUCUNE EXÉCUTION
# ------------------------------------------------------------

scanner_v3 = NexusScannerV3()

print("=== SCANNER SEGMENTÉ V3 ===")
print("Prochain segment :", scanner_v3.next_number)
print("Frontière certifiée :", scanner_v3.committed_line)
print("Source : configurée")
print("Reprise : préparée")
print("RUN_SCAN :", RUN_SCAN)
print("SCAN MASSIF : NON")
print("INSTALLATION : OK")

=== SCANNER SEGMENTÉ V3 ===
Prochain segment : 1
Frontière certifiée : 0
Source : configurée
Reprise : préparée
RUN_SCAN : False
SCAN MASSIF : NON
INSTALLATION : OK


In [34]:
import os, json, hashlib

BASE = "/kaggle/working/nexus_science_v3"
MANIFEST = os.path.join(BASE, "manifest_v3.json")
CHUNKS = os.path.join(BASE, "chunks")

print("=== AUDIT FINAL V3 ===")
print("Dossier V3 :", os.path.isdir(BASE))
print("Dossier segments :", os.path.isdir(CHUNKS))
print("Manifeste :", os.path.isfile(MANIFEST))

if os.path.isfile(MANIFEST):
    with open(MANIFEST, "r", encoding="utf-8") as f:
        m = json.load(f)

    print("\nVersion :", m.get("version"))
    print("Statut :", m.get("status"))
    print("Lignes :", m.get("scan_lines"))
    print("Notices :", m.get("selected"))
    print("Segments enregistrés :", len(m.get("completed_chunks", [])))

if os.path.isdir(CHUNKS):
    files = sorted(os.listdir(CHUNKS))
    print("\nFichiers dans chunks :", len(files))
    print("Fichiers :", files[:10])

print("\n=== SCAN MASSIF : NON ===")

=== AUDIT FINAL V3 ===
Dossier V3 : True
Dossier segments : True
Manifeste : True

Version : 3
Statut : prepared
Lignes : 0
Notices : 0
Segments enregistrés : 0

Fichiers dans chunks : 0
Fichiers : []

=== SCAN MASSIF : NON ===


In [2]:
import os

test_file = "/kaggle/working/nexus_science_v3/integration_test/test_chunk.jsonl"

if os.path.isfile(test_file):
    size = os.path.getsize(test_file)
    notices = 2093
    avg = size / notices

    # Estimation indicative, pas une prédiction garantie
    scenarios = [3_000_000, 5_000_000, 7_000_000]

    print("=== CAPACITÉ DE STOCKAGE NEXUS ===")
    print("Taille du test :", round(size / 1024**2, 2), "Mo")
    print("Taille moyenne :", round(avg, 1), "octets / notice")

    for n in scenarios:
        estimated = avg * n
        print(
            f"{n:,} notices : "
            f"{estimated / 1024**3:.2f} Go estimés"
        )

    free = 19.5
    print("\nEspace disponible initial : ~", free, "Go")
else:
    print("Fichier test introuvable.")

import os, json, shutil
from datetime import datetime, timezone

BASE = "/kaggle/working/nexus_science_v3"
PLAN = os.path.join(BASE, "storage_plan_v3.json")

os.makedirs(BASE, exist_ok=True)

free_bytes = shutil.disk_usage("/kaggle/working").free
free_gb = free_bytes / 1024**3

plan = {
    "version": 3,
    "chunk_size_notices": 100000,
    "reserve_disk_gb": 2.0,
    "available_disk_gb_at_setup": round(free_gb, 2),
    "estimated_average_bytes_per_notice": 1051,
    "estimated_notice_scenarios": [3000000, 5000000, 7000000],
    "scan_authorized": False,
    "backup_confirmed": False,
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

with open(PLAN, "w", encoding="utf-8") as f:
    json.dump(plan, f, ensure_ascii=False, indent=2)

print("=== PLAN DE STOCKAGE V3 ===")
print("Espace libre :", round(free_gb, 2), "Go")
print("Segments :", plan["chunk_size_notices"], "notices/segment")
print("Réserve minimale :", plan["reserve_disk_gb"], "Go")
print("Plan enregistré :", os.path.isfile(PLAN))
print("Scan autorisé :", plan["scan_authorized"])
print("Sauvegarde confirmée :", plan["backup_confirmed"])
print("STATUT : PRÉPARATION")

import os, json, hashlib, shutil
from datetime import datetime, timezone

BASE = "/kaggle/working/nexus_science_v3"
CHUNKS = os.path.join(BASE, "chunks")
MANIFEST = os.path.join(BASE, "manifest_v3.json")
PROTECTION = os.path.join(BASE, "protection_v3")
os.makedirs(PROTECTION, exist_ok=True)

def v3_protection_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def v3_protection_inventory():
    with open(MANIFEST, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    entries = manifest.get("completed_chunks", [])
    inventory = []

    for entry in entries:
        filename = entry.get("filename") or entry.get("file")
        if not filename:
            raise ValueError("Entrée de segment sans nom de fichier.")

        path = os.path.join(CHUNKS, filename)

        if not os.path.isfile(path):
            raise FileNotFoundError(f"Segment manquant : {filename}")

        inventory.append({
            "filename": filename,
            "bytes": os.path.getsize(path),
            "sha256": v3_protection_hash(path)
        })

    return inventory

def v3_disk_guard(required_bytes=0, reserve_gb=2):
    free = shutil.disk_usage(BASE).free
    reserve = int(reserve_gb * 1024**3)
    return {
        "free_bytes": free,
        "reserve_bytes": reserve,
        "safe": free >= required_bytes + reserve
    }

def v3_export_inventory():
    inventory = v3_protection_inventory()

    report = {
        "version": 3,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "segments": inventory,
        "segment_count": len(inventory),
        "total_bytes": sum(x["bytes"] for x in inventory),
        "backup_verified": False,
        "backup_destination": None
    }

    target = os.path.join(PROTECTION, "inventory_v3.json")
    temp = target + ".partial"

    with open(temp, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp, target)
    return report

print("=== PROTECTION V3 INSTALLÉE ===")
print("Inventaire certifiable :", callable(v3_protection_inventory))
print("Contrôle disque :", callable(v3_disk_guard))
print("Export inventaire :", callable(v3_export_inventory))
print("Destination externe : non configurée")
print("Sauvegarde réelle : non confirmée")
print("Scan massif : NON")
print("INSTALLATION : OK")

Fichier test introuvable.
=== PLAN DE STOCKAGE V3 ===
Espace libre : 19.5 Go
Segments : 100000 notices/segment
Réserve minimale : 2.0 Go
Plan enregistré : True
Scan autorisé : False
Sauvegarde confirmée : False
STATUT : PRÉPARATION
=== PROTECTION V3 INSTALLÉE ===
Inventaire certifiable : True
Contrôle disque : True
Export inventaire : True
Destination externe : non configurée
Sauvegarde réelle : non confirmée
Scan massif : NON
INSTALLATION : OK


In [1]:
!pip install -q huggingfacprint("=== FONCTIONS NEXUS DISPONIBLES ===")

for name, obj in sorted(globals().items()):
    if name.startswith("nexus_") and callable(obj):
        print(name)

print("\n=== VARIABLES NEXUS ===")

for name in sorted(globals()):
    if name.startswith(("NEXUS_", "HF_", "CHUNK_", "SCAN_")):
        obj = globals()[name]
        if not callable(obj):
            print(name, "=", str(obj)[:120])e-hub==1.32.0

SyntaxError: invalid syntax (3835190115.py, line 13)

In [82]:
from huggingface_hub import hf_hub_upload, hf_hub_download
from kaggle_secrets import UserSecretsClient
import os, hashlib

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO_ID = "Moses-team/nexus-science-backup"

V10_BACKUP_DIR = "v10_recovery_v10_3"

commit, stage_path = v10_read_commit()
stage_name = commit["stage_filename"]
remote_stage = f"{V10_BACKUP_DIR}/{stage_name}"

hf_hub_upload(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    path_or_fileobj=stage_path,
    path_in_repo=remote_stage,
    token=HF_TOKEN
)

print("=== TRANSFERT STAGE V10.3 ===")
print("Génération :", commit["generation"])
print("Fichier :", remote_stage)
print("Taille locale :", os.path.getsize(stage_path))
print("STATUT : STAGE ENVOYE")


ImportError: cannot import name 'hf_hub_upload' from 'huggingface_hub' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/__init__.py)

In [1]:
print("=== RECHARGEMENT DU MOTEUR NEXUS ===")

functions = [
    "nexus_extract_edition_json",
    "nexus_is_scientific",
    "nexus_source_identity_v6",
]

for name in functions:
    print(name, ":", callable(globals().get(name)))

print("\nFiltre scientifique :",
      "SCIENCE_KEYWORDS" in globals(),
      "EXCLUSION_KEYWORDS" in globals())

print("Base V10.2 certifiée : 3 664 notices")
print("Curseur de reprise : 15 000")
print("Scan massif : BLOQUÉ")


=== RECHARGEMENT DU MOTEUR NEXUS ===
nexus_extract_edition_json : False
nexus_is_scientific : False
nexus_source_identity_v6 : False

Filtre scientifique : False False
Base V10.2 certifiée : 3 664 notices
Curseur de reprise : 15 000
Scan massif : BLOQUÉ
